# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = os.getenv("ML_FLOW_USER", "student_your_username")
MLFLOW_PASSWORD = os.getenv("ML_FLOW_PASSWORD", "your_mlflow_password")
EXPERIMENT_NAME = os.getenv("ML_FLOW_EXPERIMENT", "")

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_melika_nobakhtian
Experiment ID: 39


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [4]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
    print("Loaded from Parquet:", parquet_path)
else:
    feature_df = pd.read_csv(csv_path)
    print("Loaded from CSV:", csv_path)
    
# fix nullable boolean columns - I got an error in part 10 because Sklearn can't handle pd.NA values.
# Add this and convert it to solve the problem
bool_cols = feature_df.select_dtypes(include="boolean").columns.tolist()
for col in bool_cols:
    feature_df[col] = feature_df[col].astype(object).where(feature_df[col].notna(), other=np.nan)

print("Fixed nullable boolean columns:", bool_cols)

# TODO: load metadata if metadata_path exists.
metadata = {}
if metadata_path.exists():
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Metadata loaded.")

print(feature_df.shape)
feature_df.head()

Loaded from Parquet: data\features\listing_availability_features_v1_student.parquet
Fixed nullable boolean columns: ['instant_bookable', 'is_superhost']
Metadata loaded.
(10480, 33)


,listing_id,neighbourhood_name,property_type,room_type,accommodates,bathrooms,bedrooms,beds,listing_price,minimum_nights,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Centrum-West,Private room in houseboat,Private room,2,1.5,1.0,1.0,132.0,3,...,0,0.000000,3.0,30.0,30,0,0.0,1,2026-04-11,v1_student
1,28871,Centrum-West,Private room in rental unit,Private room,2,1.0,1.0,1.0,89.0,2,...,13,0.419355,2.0,730.0,30,12,0.4,0,2026-04-11,v1_student
2,29051,Centrum-Oost,Private room in condo,Private room,2,1.0,1.0,1.0,61.0,2,...,12,0.387097,2.0,730.0,30,9,0.3,1,2026-04-11,v1_student
3,44391,Centrum-Oost,Entire rental unit,Entire home/apt,4,1.5,2.0,NaN,NaN,3,...,0,0.000000,3.0,730.0,30,0,0.0,1,2026-04-11,v1_student
4,48373,Buitenveldert - Zuidas,Entire rental unit,Entire home/apt,4,1.5,2.0,NaN,NaN,3,...,0,0.000000,3.0,1125.0,30,0,0.0,1,2026-04-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [5]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
# TODO: create y.
# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
# TODO: create X_clean.

assert TARGET_COL in feature_df.columns, f"Target column '{TARGET_COL}' not found!"

# Create y (target vector)
y = feature_df[TARGET_COL].astype(int)

# Clean feature columns: everything except forbidden columns
clean_feature_cols = [
    col for col in feature_df.columns
    if col not in FORBIDDEN_MODEL_COLUMNS
]

# X_clean: only clean features
X_clean = feature_df[clean_feature_cols].copy()

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.279676
1    0.720324
Name: proportion, dtype: float64
Clean feature count: 26
['neighbourhood_name', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [6]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.

leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]

X_leaky = feature_df[leaky_feature_cols].copy()

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [7]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.720300572519084
Test target rate: 0.7204198473282443


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [8]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.

numeric_cols     = X_clean.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_clean.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 21
Categorical columns: 5


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [9]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    # 2. convert scores to predictions using threshold
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    # 4. return metrics dict, y_pred, y_score

    # 1. Get probability scores
    y_score = get_positive_scores(model, X_test)

    # 2. Convert to binary predictions using threshold
    y_pred = (y_score >= threshold).astype(int)

    # 3. Compute all metrics
    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall":    recall_score(y_test, y_pred, zero_division=0),
        "f1":        f1_score(y_test, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_test, y_score),
    }

    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [10]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    # 2. save confusion_matrix.png
    # 3. save classification_report.json
    # 4. save feature_columns.json
    # 5. save dataset_metadata_snapshot.json
    
    # 1. Create run-specific folder
    run_dir = ARTIFACT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Confusion matrix PNG
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion matrix — {run_name}")
    fig.tight_layout()
    cm_path = run_dir / "confusion_matrix.png"
    fig.savefig(cm_path, dpi=100)
    plt.close(fig)
    
    # 3. Classification report JSON
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    report_path = run_dir / "classification_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)
        
    # 4. Feature columns JSON
    feature_path = run_dir / "feature_columns.json"
    with open(feature_path, "w") as f:
        json.dump({"feature_columns": feature_cols, "n_features": len(feature_cols)}, f, indent=2)
        
    # 5. Dataset metadata snapshot JSON
    meta_path = run_dir / "dataset_metadata_snapshot.json"
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)
        
    return run_dir

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [13]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    with mlflow.start_run(run_name=run_name):
        print("Step 1: run started")

        pipeline.fit(X_train, y_train)
        print("Step 2: pipeline fitted")

        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline, X_test, y_test, threshold=threshold
        )
        print("Step 3: evaluation done", metrics)

        try:
            mlflow.log_params(model_params)
            print("Step 4a: log_params OK")
        except Exception as e:
            print("Step 4a FAILED:", e)

        try:
            mlflow.log_param("threshold", threshold)
            mlflow.log_param("n_features", len(feature_cols))
            mlflow.log_param("train_size", len(X_train))
            mlflow.log_param("test_size", len(X_test))
            mlflow.log_param("dataset_version", DATASET_VERSION)
            print("Step 4b: extra log_params OK")
        except Exception as e:
            print("Step 4b FAILED:", e)

        try:
            mlflow.log_metrics(metrics)
            print("Step 5: log_metrics OK")
        except Exception as e:
            print("Step 5 FAILED:", e)

        try:
            mlflow.set_tags(tags)
            print("Step 6: set_tags OK")
        except Exception as e:
            print("Step 6 FAILED:", e)

        try:
            run_dir = save_run_artifacts(run_name, y_test, y_pred, feature_cols, metadata)
            print("Step 7a: artifacts saved locally to", run_dir)
        except Exception as e:
            print("Step 7a FAILED:", e)

        try:
            mlflow.log_artifacts(str(run_dir))
            print("Step 7b: log_artifacts OK")
        except Exception as e:
            print("Step 7b FAILED:", e)

        try:
            mlflow.sklearn.log_model(
                sk_model=pipeline,
                artifact_path="model",
                input_example=X_train.iloc[:3],
            )
            print("Step 8: log_model OK")
        except Exception as e:
            print("Step 8 FAILED:", e)

        run_id = mlflow.active_run().info.run_id
        print(f"✓ Done. Run ID: {run_id}")

    return run_id

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [14]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
# 2. build a LogisticRegression pipeline
# 3. log the run to MLflow

# Leaky train/test split
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name    = "v0_leaky_logistic_regression",
    pipeline    = leaky_pipeline,
    X_train     = X_leaky_train,
    X_test      = X_leaky_test,
    y_train     = y_leaky_train,
    y_test      = y_leaky_test,
    feature_cols= leaky_feature_cols,
    model_params= {"model": "LogisticRegression", "max_iter": 1000},
    tags        = {
        "leakage_status": "leaky",
        "known_defect":   "uses future_available_rate_30d",
        "model_family":   "logistic_regression",
    },
)

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9699427480916031, 'precision': 0.9738048461034708, 'recall': 0.9847682119205298, 'f1': 0.9792558445834705, 'roc_auc': 0.9824435503921523}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v0_leaky_logistic_regression
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 53ebcf20bc0e457db2d79772573bf445
🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/39/runs/53ebcf20bc0e457db2d79772573bf445
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


'53ebcf20bc0e457db2d79772573bf445'

## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [15]:
# TODO: build and log dummy baseline.

dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name    = "v1_dummy_baseline",
    pipeline    = dummy_pipeline,
    X_train     = X_train,
    X_test      = X_test,
    y_train     = y_train,
    y_test      = y_test,
    feature_cols= clean_feature_cols,
    model_params= {"model": "DummyClassifier", "strategy": "most_frequent"},
    tags        = {
        "leakage_status": "clean",
        "model_family":   "dummy",
    },
)

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.7204198473282443, 'precision': 0.7204198473282443, 'recall': 1.0, 'f1': 0.8374930671103716, 'roc_auc': 0.5}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v1_dummy_baseline
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: c798a1b51c424954998513363ae1cc64
🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/39/runs/c798a1b51c424954998513363ae1cc64
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


'c798a1b51c424954998513363ae1cc64'

## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [16]:
# TODO: build and log clean LogisticRegression.

lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

run_mlflow_experiment(
    run_name    = "v2_clean_logistic_regression",
    pipeline    = lr_pipeline,
    X_train     = X_train,
    X_test      = X_test,
    y_train     = y_train,
    y_test      = y_test,
    feature_cols= clean_feature_cols,
    model_params= {"model": "LogisticRegression", "max_iter": 1000, "class_weight": "None"},
    tags        = {
        "leakage_status": "clean",
        "model_family":   "logistic_regression",
    },
)

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9699427480916031, 'precision': 0.9738048461034708, 'recall': 0.9847682119205298, 'f1': 0.9792558445834705, 'roc_auc': 0.9824435503921523}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v2_clean_logistic_regression
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: d83fd80ca5594c00b7d52e1266341888
🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/39/runs/d83fd80ca5594c00b7d52e1266341888
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


'd83fd80ca5594c00b7d52e1266341888'

## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [17]:
# TODO: build and log class-weighted LogisticRegression.

lr_balanced_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

run_mlflow_experiment(
    run_name    = "v3_balanced_logistic_regression",
    pipeline    = lr_balanced_pipeline,
    X_train     = X_train,
    X_test      = X_test,
    y_train     = y_train,
    y_test      = y_test,
    feature_cols= clean_feature_cols,
    model_params= {"model": "LogisticRegression", "max_iter": 1000, "class_weight": "balanced"},
    tags        = {
        "leakage_status": "clean",
        "model_family":   "logistic_regression",
    },
)

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9699427480916031, 'precision': 0.9781890284203569, 'recall': 0.9801324503311258, 'f1': 0.9791597750578895, 'roc_auc': 0.9835036050900707}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v3_balanced_logistic_regression
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 58794b7f4d2842b1b6ea384d66726438
🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/39/runs/58794b7f4d2842b1b6ea384d66726438
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


'58794b7f4d2842b1b6ea384d66726438'

## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [18]:
# TODO: log threshold-tuning runs.

# First fit a base model to use for threshold tuning
base_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])
base_lr.fit(X_train, y_train)

for threshold in [0.30, 0.40, 0.50, 0.60]:
    run_mlflow_experiment(
        run_name    = f"v4_threshold_{str(threshold).replace('.', '')}",
        pipeline    = base_lr,
        X_train     = X_train,
        X_test      = X_test,
        y_train     = y_train,
        y_test      = y_test,
        feature_cols= clean_feature_cols,
        model_params= {
            "model":         "LogisticRegression",
            "max_iter":      1000,
            "class_weight":  "balanced",
        },
        tags        = {
            "leakage_status": "clean",
            "model_family":   "logistic_regression",
            "experiment_type":"threshold_tuning",
        },
        threshold   = threshold,
    )

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9694656488549618, 'precision': 0.9750328515111695, 'recall': 0.9827814569536424, 'f1': 0.978891820580475, 'roc_auc': 0.9835036050900707}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v4_threshold_03
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 90c54b13afa94be2b7771d1d34d24f07
🏃 View run v4_threshold_03 at: http://185.50.38.163:33014/#/experiments/39/runs/90c54b13afa94be2b7771d1d34d24f07
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39
Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9704198473282443, 'precision': 0.9769433465085638, 'recall': 0.9821192052980132, 'f1': 0.9795244385733157, 'roc_auc': 0.9835036050900707}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v4_threshold_04
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 10aed83ff1a64f6898a76c6c1b4921fa
🏃 View run v4_threshold_04 at: http://185.50.38.163:33014/#/experiments/39/runs/10aed83ff1a64f6898a76c6c1b4921fa
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39
Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9699427480916031, 'precision': 0.9781890284203569, 'recall': 0.9801324503311258, 'f1': 0.9791597750578895, 'roc_auc': 0.9835036050900707}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v4_threshold_05
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 406a975aa4a445a0aecc054e015bb679
🏃 View run v4_threshold_05 at: http://185.50.38.163:33014/#/experiments/39/runs/406a975aa4a445a0aecc054e015bb679
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39
Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9685114503816794, 'precision': 0.9794156706507304, 'recall': 0.9768211920529801, 'f1': 0.9781167108753316, 'roc_auc': 0.9835036050900707}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v4_threshold_06
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 5de4e4203de144d9bde58e5c5fbffb58
🏃 View run v4_threshold_06 at: http://185.50.38.163:33014/#/experiments/39/runs/5de4e4203de144d9bde58e5c5fbffb58
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [19]:
# TODO: build and log RandomForestClassifier.

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   RandomForestClassifier(
        n_estimators   = 200,
        max_depth      = 10,
        min_samples_leaf = 5,
        class_weight   = "balanced",
        random_state   = RANDOM_STATE,
        n_jobs         = -1,
    )),
])

run_mlflow_experiment(
    run_name    = "v5_random_forest",
    pipeline    = rf_pipeline,
    X_train     = X_train,
    X_test      = X_test,
    y_train     = y_train,
    y_test      = y_test,
    feature_cols= clean_feature_cols,
    model_params= {
        "model":            "RandomForestClassifier",
        "n_estimators":     200,
        "max_depth":        10,
        "min_samples_leaf": 5,
        "class_weight":     "balanced",
        "random_state":     RANDOM_STATE,
    },
    tags        = {
        "leakage_status": "clean",
        "model_family":   "random_forest",
    },
)

Step 1: run started
Step 2: pipeline fitted
Step 3: evaluation done {'accuracy': 0.9704198473282443, 'precision': 0.9813829787234043, 'recall': 0.9774834437086093, 'f1': 0.9794293297942933, 'roc_auc': 0.9893949325317}
Step 4a: log_params OK
Step 4b: extra log_params OK
Step 5: log_metrics OK
Step 6: set_tags OK
Step 7a: artifacts saved locally to outputs\mlflow_artifacts\v5_random_forest
Step 7b: log_artifacts OK


e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
e:\conda\envs\melika-env\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inferen

Step 8: log_model OK
✓ Done. Run ID: 772c9387bf3043c5a0d4af4c413070bf
🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/39/runs/772c9387bf3043c5a0d4af4c413070bf
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/39


'772c9387bf3043c5a0d4af4c413070bf'

## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [20]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

# Build a clean comparison table
comparison_df = runs_df[[
    "tags.mlflow.runName",
    "tags.leakage_status",
    "tags.model_family",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1",
    "metrics.roc_auc",
    "run_id",
]].copy()

comparison_df.columns = [
    "run_name", "leakage_status", "model_family",
    "accuracy", "precision", "recall", "f1", "roc_auc",
    "run_id",
]

comparison_df = comparison_df.sort_values("f1", ascending=False).reset_index(drop=True)

comparison_df

,run_name,leakage_status,model_family,accuracy,precision,recall,f1,roc_auc,run_id
0,v4_threshold_04,clean,logistic_regression,0.970420,0.976943,0.982119,0.979524,0.983504,10aed83ff1a64f6898a76c6c1b4921fa
1,v5_random_forest,clean,random_forest,0.970420,0.981383,0.977483,0.979429,0.989395,772c9387bf3043c5a0d4af4c413070bf
2,v2_clean_logistic_regression,clean,logistic_regression,0.969943,0.973805,0.984768,0.979256,0.982444,d83fd80ca5594c00b7d52e1266341888
3,v0_leaky_logistic_regression,leaky,logistic_regression,0.969943,0.973805,0.984768,0.979256,0.982444,53ebcf20bc0e457db2d79772573bf445
4,v4_threshold_05,clean,logistic_regression,0.969943,0.978189,0.980132,0.979160,0.983504,406a975aa4a445a0aecc054e015bb679
5,v3_balanced_logistic_regression,clean,logistic_regression,0.969943,0.978189,0.980132,0.979160,0.983504,58794b7f4d2842b1b6ea384d66726438
6,v4_threshold_03,clean,logistic_regression,0.969466,0.975033,0.982781,0.978892,0.983504,90c54b13afa94be2b7771d1d34d24f07
7,v4_threshold_06,clean,logistic_regression,0.968511,0.979416,0.976821,0.978117,0.983504,5de4e4203de144d9bde58e5c5fbffb58
8,v1_dummy_baseline,clean,dummy,0.720420,0.720420,1.000000,0.837493,0.500000,c798a1b51c424954998513363ae1cc64


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [21]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
BEST_RUN_ID = "772c9387bf3043c5a0d4af4c413070bf"

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Selected best run: 772c9387bf3043c5a0d4af4c413070bf


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [22]:
# TODO: replace this text.

final_explanation = """
Selected Run: v5_random_forest 

I selected the Random Forest model as the final production candidate. Although its F1 
score is marginally lower than v4_threshold_04, it achieves the 
highest ROC-AUC of all runs, meaning it has the best overall discrimination 
ability across all thresholds.

The leaky run (v0_leaky_logistic_regression) was rejected despite having identical 
metrics to v2_clean_logistic_regression. This is suspicious on its own, adding 
future_available_rate_30d should have inflated performance significantly. Regardless, this run 
must be excluded because future_available_rate_30d is derived from the label window 
and would be unavailable at inference time, making the model completely unusable in 
production.

Next steps would include hyperparameter tuning of the Random Forest via cross-validated 
grid search, exploring gradient boosting models such as XGBoost or LightGBM, and 
engineering additional features.
"""

print(final_explanation)


Selected Run: v5_random_forest 

I selected the Random Forest model as the final production candidate. Although its F1 
score is marginally lower than v4_threshold_04, it achieves the 
highest ROC-AUC of all runs, meaning it has the best overall discrimination 
ability across all thresholds.

The leaky run (v0_leaky_logistic_regression) was rejected despite having identical 
metrics to v2_clean_logistic_regression. This is suspicious on its own, adding 
future_available_rate_30d should have inflated performance significantly. Regardless, this run 
must be excluded because future_available_rate_30d is derived from the label window 
and would be unavailable at inference time, making the model completely unusable in 
production.

Next steps would include hyperparameter tuning of the Random Forest via cross-validated 
grid search, exploring gradient boosting models such as XGBoost or LightGBM, and 
engineering additional features.

